# Qiskit Fall Fest 2026, Session 3: Quantum cryptography

Wed, Nov 18, 2026, 5:00 to 6:00 PM ET, on Zoom

Covers: No-cloning, BB84 key exchange with an eavesdropper, and post-quantum cryptography.

This is the third of five sessions this week. Tonight has three parts.

- Seminar: why encryption is at risk, and what physics offers.
- Lab: BB84 key exchange, with and without an eavesdropper.
- Challenge 3 opens at 6:00 PM. Every earlier challenge stays open until Sun, Nov 22 at 11:59 PM, so you can still take part in all of them if you join tonight.

No cryptography background is assumed. The words in the "Covers" line and in the list above, in one sentence each. Cryptography is the study of how to keep messages secret. Encryption scrambles a message so that only someone who holds the right key can read it, and a key is the string of bits, 0s and 1s, that locks and unlocks the message. A key exchange is the way two people agree on a secret key when everything they send can be overheard. An eavesdropper is someone who listens in on the line between them. BB84 is a key exchange that sends the key as qubits, the quantum version of bits. Its name stands for its inventors, Charles Bennett and Gilles Brassard, and the year 1984. No-cloning is the rule of quantum physics that an unknown qubit cannot be copied. Post-quantum cryptography is encryption that runs on the computers we already have and is built to hold up against a quantum computer. Qiskit is the open-source Python library, developed by IBM, that you use all week to write quantum programs and run them.

This notebook stands on its own. If you missed Sessions 1 and 2, start here anyway: Part 1 opens with the few ideas that tonight needs, most of them from Monday, and every other term is explained where it first appears.

Tonight's whole lab runs on a simulator, a program that imitates a quantum computer. It runs on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Nothing in this notebook needs an account, and no cell sends anything to a real quantum computer. The Setup section below is the same in all five notebooks, so it speaks of processors and of hardware cells. A processor is the chip that holds the qubits, and a hardware cell is a cell that can run on a real one. This notebook has no hardware cell.

## What you will be able to do

- Say why measuring a qubit in the wrong basis disturbs it, and why an unknown qubit cannot be copied. To measure a qubit is to read it out, which gives a 0 or a 1. A basis is a choice of how to prepare and how to measure a qubit. Tonight there are two such choices, called Z and X.
- Run the BB84 key exchange as circuits and sift a shared key. A circuit is a list of operations applied to qubits, usually ending in a measurement. To sift is to keep only the positions where the sender and the receiver happened to use the same basis.
- Add an eavesdropper and measure the error rate she causes. The error rate is the fraction of the key in which the receiver's bit differs from the sender's.
- Decide from an error rate whether an exchange was intercepted.
- Tell quantum key distribution apart from post-quantum cryptography. Quantum key distribution, QKD for short, is the general name for key exchanges that send qubits, such as BB84.

## How to use this notebook

- Run the cells in order, from the top. A cell is one box of text or code. To run a code cell, click it and press Shift+Enter.
- Part 1 follows tonight's slides. Each idea has a short recap, and often a small cell you can run to see the idea at work.
- Part 2 is the lab. The code under "Lab" is complete. Run it, read the output, then change it and run it again.
- After each lab there is a "Your turn" exercise. Replace the line `# YOUR CODE HERE` with your own code, run the cell, then run the check cell below it. A check that fails stops with a red error message. Its last line is the hint. Run all (Colab: Runtime menu; Jupyter: Run menu) stops at the first check whose exercise you have not done yet. Do the exercise there, or run the cells after it one by one.
- Everything runs on a simulator, on the computer that runs this notebook, whatever the switches in the Setup section say. This notebook has no hardware cell.
- Use this notebook during the session and again afterwards on your own.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## Join the Discord and register

*Slide 2.*

- **Discord.** Join the server at https://discord.gg/FBNzRcxMr. Questions between sessions go there.
- **Registration.** Not registered yet? Fill in the form at https://forms.gle/NC9o1iRk3WggLuTH7.

On the slide the two links are QR codes, small square patterns that a phone camera reads as a link. Discord is the chat service where the club answers questions.

## New tonight?

*Slide 4.*

- **IBM Quantum Platform.** quantum.cloud.ibm.com. Open Plan instance and an API key. This is where your circuits run.
- **qBraid, the second option.** account.qbraid.com. Sign up with the email you registered with. Access is sent there.
- **Step by step.** The Accounts section of the Session 1 notebook, on the event page.

Tonight's lab needs no account. Open tonight's notebook and run the first cell.

You need an account only to run on a real quantum computer, and tonight nothing does. This notebook has no hardware cell, so every cell runs on the simulator tonight, whatever the three switches in the Setup section say. Leave them as they are, do the whole lab on the simulator, and set up an account later for the sessions that use one.

Some words from the first card. IBM Quantum Platform is the website where you manage your IBM Quantum account. An API key is a secret string of characters that lets a program, such as a notebook, act for your account. A plan is the kind of access an account has, and the Open Plan is the one you do not pay for. An instance is your share of access under a plan.

IBM may ask for a credit card when you create the account. The card is not charged unless you create a Pay-As-You-Go instance and use it to send work to a quantum computer. If you cannot or do not want to give one, ask an organizer on Discord. Challenges 3 and 4 need no account.

In the Accounts section of the Session 1 notebook the steps are cells you can run: IBM Quantum Platform first, then the second option. A job is one request to run circuits, and a processor is the chip that holds the qubits. Jobs on IBM processors always go through your own IBM Quantum account. The second option does not replace that account. It adds the other quantum computers that qBraid offers.

In this notebook "the first cell" means the three code cells of the Setup section above. If they ran without an error, you are ready.

## Part 1: The seminar ideas

This part follows tonight's slides in order. Each heading is the title of a slide. Under it you find a short recap, the math from the slide, and often a small cell to run. The sections called "Derivation" show the steps behind a result, one at a time.

**If you missed Sessions 1 and 2.** Tonight needs a few ideas from Monday, plus two that are new: measuring in the X basis and the overlap of two states. Here they are, each in a sentence or two.

- **Qubit.** The quantum version of a bit. When you measure a qubit, you read a 0 or a 1.
- **State and ket.** The state of a qubit is its full description: everything you need in order to predict what a measurement will give. $\lvert 0 \rangle$ and $\lvert 1 \rangle$ are the two states in which a measurement is certain to give 0 or 1. The bracket around the label is called a ket. It marks the label as the name of a state.
- **Superposition and amplitude.** A general state is a blend of the two, written $\alpha\lvert 0 \rangle + \beta\lvert 1 \rangle$. The numbers $\alpha$ and $\beta$ are called amplitudes. A state in which both are different from zero is a superposition. Tonight's amplitudes are ordinary numbers, and some of them are negative.
- **Probability.** Measure that state and you read 0 with probability $\alpha^2$ and 1 with probability $\beta^2$. The two probabilities add up to 1. (In general an amplitude can be a complex number, and the rule then uses its squared size. Tonight does not need that.)
- **Plus and minus.** Two superpositions have names of their own: $\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$ and $\lvert - \rangle = \frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$. In both, each amplitude squared is one half, so a measurement gives 0 or 1 like a fair coin.
- **Gate.** An operation that changes the state of a qubit without measuring it. Almost everything tonight uses two gates. A third, CX, appears in one place, under "You cannot copy an unknown qubit", and is explained there. X flips a qubit: it turns $\lvert 0 \rangle$ into $\lvert 1 \rangle$ and $\lvert 1 \rangle$ into $\lvert 0 \rangle$. H, the Hadamard gate, turns $\lvert 0 \rangle$ into $\lvert + \rangle$ and $\lvert 1 \rangle$ into $\lvert - \rangle$. H undoes itself, so a second H turns $\lvert + \rangle$ back into $\lvert 0 \rangle$ and $\lvert - \rangle$ back into $\lvert 1 \rangle$.
- **Measurement and basis.** A measurement asks a qubit a question that has two answers. The pair of states that give the two answers with certainty is called a basis. The ordinary measurement asks "0 or 1?". Its basis is $\lvert 0 \rangle$, $\lvert 1 \rangle$, and it is called the Z basis. Tonight's other question is "plus or minus?". Its basis is $\lvert + \rangle$, $\lvert - \rangle$, and it is called the X basis. The measurement built into Qiskit is the Z one, so an X measurement takes two steps: apply H, then measure. Reading 0 then means plus, and reading 1 means minus. A measurement also changes the qubit: afterwards it is in the state that goes with the answer. After a Z measurement that is $\lvert 0 \rangle$ if you read 0 and $\lvert 1 \rangle$ if you read 1.
- **Overlap.** $\langle a \vert b \rangle$ is a number that says how alike the states $\lvert a \rangle$ and $\lvert b \rangle$ are. For tonight's states you get it by multiplying the two amplitudes of 0, multiplying the two amplitudes of 1, and adding the two products. For example, $\langle 0 \vert + \rangle = 1 \times \frac{1}{\sqrt{2}} + 0 \times \frac{1}{\sqrt{2}} = \frac{1}{\sqrt{2}}$. A state has overlap 1 with itself, and the two states of one basis have overlap 0 with each other. The squared overlap $\lvert \langle a \vert b \rangle \rvert^2$ is the probability that a qubit prepared in $\lvert b \rangle$ is found in $\lvert a \rangle$ by a measurement in the basis that $\lvert a \rangle$ belongs to.
- **Circuit and shot.** A circuit is a list of gates applied to qubits, usually ending in a measurement. One run of a circuit is called a shot.

The cell below builds the four states that tonight is about, $\lvert 0 \rangle$, $\lvert 1 \rangle$, $\lvert + \rangle$ and $\lvert - \rangle$, from the two gates. A `Statevector` holds the list of amplitudes of a state. For each state the cell prints the two amplitudes and the two probabilities.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

zero = QuantumCircuit(1)       # a circuit with one qubit. Every qubit starts in 0.
one = QuantumCircuit(1)
one.x(0)                       # X flips 0 to 1
plus = QuantumCircuit(1)
plus.h(0)                      # H turns 0 into plus
minus = QuantumCircuit(1)
minus.x(0)
minus.h(0)                     # X, then H: H turns 1 into minus

for name, circuit in [("0", zero), ("1", one), ("+", plus), ("-", minus)]:
    amplitudes = Statevector(circuit).data.real          # the amplitudes of 0 and of 1, as ordinary numbers
    print("state", name, "| amplitudes", np.round(amplitudes, 3),
          "| probabilities of reading 0 and 1:", np.round(amplitudes ** 2, 3))

Each line agrees with the list above. 0.707 is $\frac{1}{\sqrt{2}}$, rounded to three decimals. The states $\lvert + \rangle$ and $\lvert - \rangle$ give the same two probabilities and differ only in one sign. A Z measurement cannot see that sign. An X measurement can: it tells the two apart with certainty.

### Today's public-key encryption rests on problems a quantum computer can solve

*Slide 5.*

- **RSA.** Rests on factoring large numbers.
- **Elliptic curves.** Rest on discrete logarithms.
- **Shor's algorithm.** Solves both in polynomial time, on a large enough machine.

No machine today is large enough. Traffic recorded today can be decrypted later.

RSA and elliptic curves are the two kinds of public-key encryption on this slide. In public-key encryption every person has two keys. One is public: anyone may use it to lock a message for that person, or to agree on a secret key with them. The other is private, and only it unlocks. The two keys are tied together by a math problem that is quick in one direction and far too slow in the other on a classical computer. A classical computer is any ordinary computer, from a phone to a supercomputer. An eavesdropper who could do the slow direction could work out the private key from the public one.

For RSA the problem is factoring. Multiplying two large prime numbers takes a computer a moment. Starting from the product and finding the two primes again is the slow direction. For elliptic curves the problem is the discrete logarithm. There you are shown the result of repeating one operation a secret number of times, and you have to find that number. Computing the result is quick. Finding the number of repetitions from the result is the slow direction.

Shor's algorithm is a quantum algorithm, a recipe whose steps run on qubits. It solves both problems in polynomial time, which means that its number of steps grows only like a fixed power of the size of the key. Its central step is the quantum Fourier transform from Tuesday's session, a small circuit that turns a repeating pattern in the amplitudes into a result that can be measured. The derivation below puts the costs in symbols.

To be exact about where things stand: no quantum computer today is large enough to do this to real keys. The reason to act now is the last line of the slide: encrypted traffic that someone records today can be stored and decrypted later, once a large enough machine exists.

Not all encryption is hit equally. In a symmetric cipher the sender and the receiver share one secret key, which both locks and unlocks. AES, the Advanced Encryption Standard, is such a cipher, and AES-256 is AES with a key of 256 bits. Against a symmetric cipher a quantum computer can use Grover's search, also from Tuesday, which only halves the effective key length. AES-256 keeps about 128 bits of security, where $k$ bits of security means that an attack needs about $2^k$ steps. The last two lines of the derivation below show where the 128 comes from.

#### Derivation: What Shor's algorithm changes, in symbols

*Slide 10.*

Call $N$ the large number in an RSA public key, the product of the two secret primes. The size of the key is the number of bits of $N$. A 2,048-bit key has $N \approx 2^{2048}$, a number with 617 decimal digits. The logarithms below measure that size: $\ln N$, the natural logarithm of $N$, is proportional to the number of digits. Shor's line below writes $\log N$ without a base. Logarithms in different bases differ only by a constant factor, so every one of them is proportional to the number of digits, and the plot below uses the number of bits.

Best known classical factoring. The fastest known method for a classical computer is called the number field sieve. Its number of steps grows like the expression below, with $c$ about 1.92. Here $\exp(x)$ means $e^{x}$.

$$\exp\left( c\,(\ln N)^{1/3}(\ln\ln N)^{2/3} \right)$$

The exponent grows with the size of the key, though more slowly than the size itself. Growth of this kind is called sub-exponential: slower than growth by any fixed factor with every extra digit, and still faster than any fixed power of the number of digits. It is still far too slow for 2,048-bit keys.

Shor's algorithm, roughly. Its number of steps grows like the cube of the size of the key.

$$O((\log N)^3)$$

The notation $O(\ldots)$ means "grows no faster than a constant times". This is what polynomial time looks like: a fixed power, here the third, of the number of digits. The word "roughly" matters. The exact count depends on how the arithmetic inside the algorithm is built, and the cube is an order of growth, not a run time.

Grover against a $k$-bit key. A key of $k$ bits is one of $2^k$ possible keys, and trying them one after another takes up to $2^k$ steps. Grover's search finds the one right key among all $2^k$ in about $\sqrt{2^k} = 2^{k/2}$ steps.

$$2^k \rightarrow 2^{k/2} \text{ steps}$$

So AES-256 still needs $2^{128}$ steps: against that attack it keeps the strength of a 128-bit key.

$$2^{256/2} = 2^{128} \text{ steps}$$

Halving the exponent is what "halves the effective key length" means. Compare the two attacks. For factoring, Shor's algorithm replaces a sub-exponential cost by a polynomial one. On a large enough machine a longer RSA key would help only a little, and the same goes for elliptic curves. Grover's search only takes a square root, and a symmetric cipher answers it with a longer key.

The cell below plots the first two formulas against the key size, from 256 to 4,096 bits. It keeps $c = 1.92$ and drops every other constant, so it shows the shape of the two curves and not real run times. The vertical axis is logarithmic: equal distances on it mean equal factors.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

key_bits = np.arange(256, 4097, 64)                 # key sizes from 256 to 4,096 bits
ln_N = key_bits * np.log(2)                         # N is about 2**bits, so ln N = bits * ln 2
classical = np.exp(1.92 * ln_N ** (1 / 3) * np.log(ln_N) ** (2 / 3))   # the first formula, with c = 1.92
shor = key_bits.astype(float) ** 3                  # the second formula: the number of bits, cubed

at_2048 = list(key_bits).index(2048)
print("at 2,048 bits the classical formula gives about 10^" + str(round(np.log10(classical[at_2048]))),
      "and the cubic about 10^" + str(round(np.log10(shor[at_2048]))))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(key_bits, classical, color="tab:red", label="best known classical factoring")
ax.semilogy(key_bits, shor, color="tab:blue", label="Shor's algorithm, roughly")
ax.axvline(2048, color="0.6", ls=":")
ax.annotate("2,048-bit key", (2048, 1e20), textcoords="offset points", xytext=(6, 0))
ax.set_xlabel("key size (number of bits of N)")
ax.set_ylabel("steps (shape only)")
ax.set_title("How the cost of factoring grows with the key size")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.show()

At 2,048 bits the classical formula gives about $10^{35}$ and the cubic about $10^{10}$. The red curve, best known classical factoring, keeps climbing as the key grows. The blue curve, Shor's algorithm, is nearly flat on this scale. A longer key therefore buys a lot against the classical attack and almost nothing against Shor's algorithm.

Read these as orders of growth, not as run times. A real attack has large constant factors on both curves, and the quantum computer that the blue curve needs does not exist today.

### Measuring a qubit changes it

*Slide 11.*

- **Prepare.** Alice sends $\lvert + \rangle$: an equal superposition of 0 and 1 in the Z basis, and a certain $+$ in the X basis.
- **Look the wrong way.** A Z measurement returns 0 or 1 at random and leaves the qubit in that state.
- **The trace it leaves.** An X measurement that was certain to give $+$ is now a coin flip.

Measuring in the right basis changes nothing. Only the wrong basis disturbs, and Alice and Bob can check for it.

In cryptography the sender is traditionally called Alice and the receiver Bob. Here Alice prepares a qubit in the state

$$\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$$

Asked "plus or minus?", this qubit answers plus every time. Asked "0 or 1?", it answers at random.

Now someone looks at the qubit in the wrong basis. A Z measurement gives 0 or 1 with probability $\frac{1}{2}$ each, and it leaves the qubit in $\lvert 0 \rangle$ or in $\lvert 1 \rangle$. The plus state is gone. Whichever of the two is left, an X measurement on it gives plus with probability

$$\lvert \langle + \vert 0 \rangle \rvert^2 = \lvert \langle + \vert 1 \rangle \rvert^2 = \frac{1}{2}$$

Before the Z measurement that probability was 1. So a look in the wrong basis leaves a trace: an X measurement that should have been certain now comes out wrong half of the time. A look in the right basis, here X, reads plus and leaves the qubit exactly as it was. This disturbance, together with the no-cloning rule on the next slide, is what BB84 uses to catch an eavesdropper.

The cell below defines `plus_then_x`, a small function that builds this experiment as a circuit, and draws the version with the Z measurement. Read the drawing from left to right. The line `q` is the qubit, each box is a gate, and each dial is a measurement. The double lines `mid` and `out` are classical registers, named groups of ordinary bits that store what the measurements read. Remember that an X measurement is an H followed by a measurement.

In [ ]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister

def plus_then_x(z_first):
    """Prepare plus, measure in the Z basis if z_first is True, then measure in the X basis."""
    mid = ClassicalRegister(1, "mid")      # one ordinary bit for the Z measurement in the middle
    out = ClassicalRegister(1, "out")      # one ordinary bit for the X measurement at the end
    circuit = QuantumCircuit(QuantumRegister(1, "q"), mid, out)
    circuit.h(0)                   # prepare plus
    if z_first:
        circuit.measure(0, mid)    # the Z measurement in the middle
    circuit.h(0)                   # X measurement, step 1: H
    circuit.measure(0, out)        # X measurement, step 2: measure. Reading 0 means plus.
    return circuit

plus_then_x(True).draw("mpl")

The next cell runs the experiment both ways, 1,000 shots each: first without the Z measurement, then with it. A sampler is the Qiskit tool that runs circuits for a number of shots and returns the measured bits. This circuit has a measurement in the middle, so the cell uses the sampler from Aer, Qiskit's simulator package, which can handle that. Aer's sampler class is called `SamplerV2`, and the import gives it the name `AerSampler`, which says where it comes from. Look at how often the X measurement at the end gives plus.

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSampler

shots = 1000
aer_sampler = AerSampler(seed=2026)        # the seed fixes the random numbers, so the text below stays true
job = aer_sampler.run([plus_then_x(False), plus_then_x(True)], shots=shots)
no_look, z_look = job.result()             # one result for each of the two circuits

print("no Z measurement:  plus in", no_look.data.out.get_counts().get("0", 0), "of", shots, "shots")
print("Z measured first:  plus in", z_look.data.out.get_counts().get("0", 0), "of", shots, "shots")
print("the Z measurement in the middle read 0 in", z_look.data.mid.get_counts().get("0", 0), "of", shots, "shots")

Without the Z measurement, all 1,000 shots gave plus. With the Z measurement in the middle, only 525 of the 1,000 did, which is about half. The Z measurement itself read 0 in 491 shots and 1 in the rest. So whoever looked in the wrong basis learned nothing about the state that was sent, and still spoiled it for the measurement that came after.

### You cannot copy an unknown qubit

*Slide 12.*

$$\text{No single } U \text{ gives} \quad U \lvert \psi \rangle \lvert 0 \rangle = \lvert \psi \rangle \lvert \psi \rangle \quad \text{for every } \lvert \psi \rangle$$

No single operation $U$ copies every state.
A copier that works on $\lvert 0 \rangle$ and $\lvert 1 \rangle$ must fail on $\lvert + \rangle$.

How to read the line. $\lvert \psi \rangle$ stands for any state of one qubit (the Greek letter is called psi). Two kets side by side describe two qubits: $\lvert \psi \rangle \lvert 0 \rangle$ is one qubit in the state $\lvert \psi \rangle$ next to a second, blank qubit in $\lvert 0 \rangle$. $U$ stands for any operation on the two qubits that is built from gates, with no measurement in it. A copier would be a $U$ that turns the blank into a second $\lvert \psi \rangle$ and leaves the first qubit as it was, whatever $\lvert \psi \rangle$ is. The line says that no such $U$ exists. This is called the no-cloning theorem.

Read the words "for every" with care. Some states can be copied. The CX gate, also called CNOT, acts on two qubits: it flips the second one, the target, if the first one, the control, is 1. With a blank target in $\lvert 0 \rangle$ it copies a 0 or a 1 from the control to the target. What is impossible is one machine that copies every state. In particular no machine copies both the states of the Z basis and the states of the X basis.

Why this matters tonight. An ordinary bit can be copied as often as you like, so an eavesdropper on an ordinary line can keep a perfect record and leave no trace. With qubits she cannot keep a copy and pass the original along. She has to measure the original, and the previous section, "Measuring a qubit changes it", showed what a measurement in the wrong basis does.

The cell below tries CX as a copier. Qubit 0 is the original and qubit 1 is the blank. The cell first puts the original in 0 and then in 1, and prints the probabilities of the four possible results. Then it feeds the same copier the original $\lvert + \rangle$ and compares what comes out with two true copies of $\lvert + \rangle$. Each printed list has one entry for each of the results 00, 01, 10 and 11, in that order.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# First: the copier on the originals 0 and 1.
for bit in (0, 1):
    copier = QuantumCircuit(2)       # qubit 0 is the original, qubit 1 is the blank, which starts in 0
    if bit:
        copier.x(0)                  # put the original in 1
    copier.cx(0, 1)                  # CX: flip the blank if the original is 1
    probabilities = Statevector(copier).probabilities()
    print("original", bit, "| probabilities of reading 00, 01, 10, 11:", probabilities)

# Then: the same copier on the original plus.
copier = QuantumCircuit(2)
copier.h(0)                          # put the original in plus
copier.cx(0, 1)
made = Statevector(copier)
wanted = Statevector.from_label("++")                    # two copies of plus
print("the copier made,    amplitudes of 00, 01, 10, 11:", np.round(made.data.real, 3))
print("two copies of plus, amplitudes of 00, 01, 10, 11:", np.round(wanted.data.real, 3))
print("squared overlap between the two:", round(abs(made.inner(wanted)) ** 2, 3))

With the original in 0 the result is 00 with certainty, and with the original in 1 it is 11. Both qubits hold the bit, so the copy worked. (Qiskit prints qubit 0 as the right character. In 00 and 11 the two characters are equal, so the order does not show.)

With the original in $\lvert + \rangle$ the copier made a state with the amplitude 0.707 on 00 and on 11:

$$\frac{\lvert 00 \rangle + \lvert 11 \rangle}{\sqrt{2}}$$

Two copies of $\lvert + \rangle$ would have the amplitude 0.5 on all four results:

$$\lvert + \rangle \lvert + \rangle = \frac{\lvert 00 \rangle + \lvert 01 \rangle + \lvert 10 \rangle + \lvert 11 \rangle}{2}$$

The squared overlap between the two is 0.5, so what the copier made is not the copy. It is a different state, called a Bell pair. Its two qubits are entangled: measured in the Z basis they always give the same bit, yet neither of them is in the state $\lvert + \rangle$ by itself. The reason is that a gate acts on each part of a superposition separately. CX turned the 0 part into 00 and the 1 part into 11, and that is all it can do.

#### Derivation: No-cloning, in four lines

*Slide 13.*

The cell above showed one copier that fails. The argument below shows that every copier must fail. It needs two facts about overlaps.

- An operation $U$ built from gates never changes the overlap between two states. Whatever overlap two states have before $U$, they have the same overlap after it. A gate moves states the way a rotation moves arrows: every arrow turns, and the angles between them stay as they were. An operation with this property is called unitary, and every gate is one.
- For two qubits that each have a state of their own, overlaps multiply qubit by qubit: the overlap between $\lvert a \rangle \lvert b \rangle$ and $\lvert c \rangle \lvert d \rangle$ is $\langle a \vert c \rangle \langle b \vert d \rangle$.

Suppose $U$ copies two states, $\lvert \psi \rangle$ and $\lvert \varphi \rangle$ (psi and phi).

$$U\lvert \psi \rangle\lvert 0 \rangle = \lvert \psi \rangle\lvert \psi \rangle, \quad U\lvert \varphi \rangle\lvert 0 \rangle = \lvert \varphi \rangle\lvert \varphi \rangle$$

$U$ keeps the overlap of two states. In math words: unitaries preserve inner products, and inner product is the math name for the overlap. Before the copier runs, the two inputs have the overlap $\langle \psi \vert \varphi \rangle \langle 0 \vert 0 \rangle$: one factor from the two originals and one from the two blanks. After it has run, the two outputs have the overlap $\langle \psi \vert \varphi \rangle \langle \psi \vert \varphi \rangle$, one factor from each qubit. The first fact says that the two numbers are equal.

$$\langle \psi \vert \varphi \rangle\langle 0 \vert 0 \rangle = \langle \psi \vert \varphi \rangle^2$$

So the overlap is 0 or 1. A state has overlap 1 with itself, so $\langle 0 \vert 0 \rangle = 1$ and the left side is $\langle \psi \vert \varphi \rangle$ alone. Call that number $x$. The equation then reads $x = x^2$, which is the same as $x(1 - x) = 0$. Its only solutions are $x = 0$ and $x = 1$.

$$\langle \psi \vert \varphi \rangle = \langle \psi \vert \varphi \rangle^2 \; \Rightarrow \; \langle \psi \vert \varphi \rangle \in \lbrace 0, 1 \rbrace$$

Overlap 1 means that the two states are the same state. Overlap 0 means that they are orthogonal: one measurement tells them apart with certainty, like $\lvert 0 \rangle$ and $\lvert 1 \rangle$. So the only states that one machine can copy are identical or orthogonal.

But these two states overlap.

$$\langle 0 \vert + \rangle = \frac{1}{\sqrt{2}}$$

That is about 0.707, which is neither 0 nor 1. So no machine copies both $\lvert 0 \rangle$ and $\lvert + \rangle$. The same holds for any state of the Z basis paired with any state of the X basis: the two are neither identical nor orthogonal. That is the core of the security argument of BB84. An eavesdropper who does not know in which basis a qubit was prepared cannot copy it.

### BB84: send in random bases, keep the matches

*Slide 14.*

1. Alice picks a random bit and a random basis for each qubit: Z ($\lvert 0 \rangle$, $\lvert 1 \rangle$) or X ($\lvert + \rangle$, $\lvert - \rangle$).
2. Bob measures each qubit in his own random basis.
3. They compare bases in public and keep only the positions that match, about half.

BB84 turns the last two ideas, disturbance and no-cloning, into a protocol for sharing a key. A protocol is a fixed sequence of steps that both sides follow. Alice wants Bob to end up with the same string of random bits that she has, and nobody else should know it. She sends the bits one qubit at a time.

For each qubit Alice flips two coins. The first coin is the bit, 0 or 1. The second is the basis. In the Z basis she sends $\lvert 0 \rangle$ for the bit 0 and $\lvert 1 \rangle$ for the bit 1. In the X basis she sends $\lvert + \rangle$ for the bit 0 and $\lvert - \rangle$ for the bit 1. Bob does not know her bases, so for each qubit he flips a coin of his own and measures in that basis.

When the two bases match, Bob's bit equals Alice's with certainty. When they differ, his result is a coin flip and tells him nothing. After all the qubits have arrived, Alice and Bob tell each other which basis they used for each qubit. They do this over an ordinary line that anyone may listen to. In this step they do not say which bit they sent or read. Then both keep the positions where the bases match and throw the rest away. This step is called sifting, and the bits that remain are the sifted key. About half of the positions survive, because two fair coins agree half of the time.

The bases are public. The bits of the key never are. Announcing the bases is harmless because it happens after every qubit has been measured: by then nobody can use the announcement to choose the right basis for a measurement.

The picture on the slide shows a BB84 exchange as a Qiskit circuit, with Alice, Eve and Bob marked along the wire, a padlock, and four bits of which two are kept. Eve is the usual name for the eavesdropper. The cell below draws the circuit for one qubit, with nobody listening. Alice sends the bit 1 in the X basis, and Bob measures in the X basis. The dashed line is a barrier. It does nothing to the qubit: it is a separator in the drawing, and here it marks the trip from Alice to Bob.

In [ ]:
from qiskit import QuantumCircuit

one_qubit = QuantumCircuit(1, 1)     # one qubit, and one ordinary bit to hold the measured result
one_qubit.x(0)                       # Alice encodes the bit 1 ...
one_qubit.h(0)                       # ... in the X basis: the qubit is now in the state minus
one_qubit.barrier()                  # a separator in the drawing: the qubit travels from Alice to Bob
one_qubit.h(0)                       # Bob chose the X basis: H ...
one_qubit.measure(0, 0)              # ... then measure
one_qubit.draw("mpl")

Read the drawing with the rules you already have. X encodes the bit 1. The first H moves it into the X basis. The second H, right before the dial, turns Bob's measurement into an X measurement. Here both chose X, so the two H gates undo each other and Bob reads 1 for certain. Lab 1 builds one such circuit for every qubit.

#### Derivation: The four BB84 states

*Slide 15.*

Z basis. The bit 0 is sent as the first of these two states and the bit 1 as the second.

$$\lvert 0 \rangle, \quad \lvert 1 \rangle$$

X basis. Again the bit 0 is sent as the first state and the bit 1 as the second.

$$\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}, \quad \lvert - \rangle = \frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$$

Same basis: certain. The squared overlap is the probability that a qubit prepared in the state on the right is read as the state on the left. Say Alice sends $\lvert 0 \rangle$ and Bob measures in the Z basis. He reads 0 with probability 1 and 1 with probability 0.

$$\lvert \langle 0 \vert 0 \rangle \rvert^2 = 1, \quad \lvert \langle 1 \vert 0 \rangle \rvert^2 = 0$$

Other basis: a coin flip. Say Alice sends $\lvert + \rangle$ and Bob measures in the Z basis. In $\lvert + \rangle$ the amplitudes of 0 and of 1 are both $\frac{1}{\sqrt{2}}$, so both overlaps are $\frac{1}{\sqrt{2}}$ and both probabilities are one half.

$$\lvert \langle 0 \vert + \rangle \rvert^2 = \lvert \langle 1 \vert + \rangle \rvert^2 = \frac{1}{2}$$

These four numbers are the entire protocol. Matching bases give certainty. Mismatched bases give no information.

The derivation works out one case of each kind. The cell below computes the squared overlap for every case: each of the four states that Alice can send, read as each of the four states.

In [ ]:
from qiskit.quantum_info import Statevector

names = ["0", "1", "+", "-"]
print("sent    P(read 0)  P(read 1)  P(read +)  P(read -)")
for sent in names:
    row = "  " + sent + "   "
    for read in names:
        overlap = Statevector.from_label(read).inner(Statevector.from_label(sent))   # the number <read|sent>
        row += f"{abs(overlap) ** 2:11.2f}"
    print(row)

Read the table one row at a time. A measurement uses one basis, so look at the two Z columns together and at the two X columns together. Inside the basis that the state was sent in, the entries are 1 and 0: the result is certain. In the other basis both entries are 0.5: a coin flip. The four numbers of the derivation are the Z columns of the first row and of the third row.

### Eight qubits, by hand

*Slide 16.*

| Qubit | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| Alice's bit | 1 | 0 | 0 | 1 | 1 | 0 | 1 | 0 |
| Alice's basis | Z | X | X | Z | X | Z | Z | X |
| Bob's basis | Z | Z | X | X | X | Z | X | Z |
| Bob's result |  |  |  |  |  |  |  |  |
| Keep? |  |  |  |  |  |  |  |  |

Fill in the last two rows on paper before you run the cell below. Two rules are enough. Where the two bases match, Bob's result is Alice's bit, and the position is kept. Where they differ, Bob's result is a coin flip, and the position is discarded. Then read off the shared key: the kept bits, from left to right.

The cell below applies the two rules to one qubit after the other. Where the bases differ it prints a question mark for Bob's result, because that result is random.

In [ ]:
table_bits = [1, 0, 0, 1, 1, 0, 1, 0]                           # Alice's bits
table_alice = ["Z", "X", "X", "Z", "X", "Z", "Z", "X"]          # Alice's bases
table_bob = ["Z", "Z", "X", "X", "X", "Z", "X", "Z"]            # Bob's bases

shared_key = ""
position = 0
for bit, alice_basis, bob_basis in zip(table_bits, table_alice, table_bob):   # walk through the three rows together
    position += 1
    if alice_basis == bob_basis:      # same basis: Bob reads Alice's bit, and the position is kept
        print("qubit", position, "| bases", alice_basis, bob_basis, "| Bob's result:", bit, "| keep: yes")
        shared_key += str(bit)
    else:                             # different bases: a coin flip, and the position is discarded
        print("qubit", position, "| bases", alice_basis, bob_basis, "| Bob's result: ? | keep: no")
print("shared key:", shared_key)

Compare the printed lines and the key with what you wrote down. Lab 1 does the same for 64 qubits, with a circuit for every qubit in place of the two rules.

### An eavesdropper leaves a 25% error rate

*Slide 25.*

$$\text{QBER} = \frac{1}{2} \times \frac{1}{2} = \frac{1}{4}$$

QBER is the quantum bit error rate. Eve guesses the wrong basis half the time. When she does, Bob's bit is wrong half the time.

The quantum bit error rate is the fraction of the sifted key in which Bob's bit differs from Alice's. With nobody listening, and on ideal devices, it is zero.

Now Eve sits on the line. No-cloning rules out keeping a copy of each qubit, so she does the next best thing. She measures each qubit as it passes and sends a fresh qubit on to Bob, prepared in the state that she read. This is called the intercept-and-resend attack. Eve has the same problem as Bob: she does not know Alice's basis, so she has to guess one.

Take a kept position, one where Alice and Bob used the same basis. If Eve guessed that basis too, which happens with probability one half, she reads the right bit, resends the same state, and causes no error. If she guessed the other basis, also with probability one half, the qubit she resends is in a state of her own basis. Seen from the basis of Alice and Bob, that state is an equal superposition, so Bob reads the wrong bit with probability one half. The two halves multiply: one position in four of the sifted key is wrong.

The 25% holds under two conditions. Eve intercepts every qubit, and the line and the devices add no errors of their own. If she listens to only some of the qubits, the rate is lower in proportion, and "Your turn 2" in Part 2 tries that. Errors that come from the devices are the subject of a later slide, "Noise looks like a small eavesdropper".

#### Derivation: Where the 25% comes from

*Slide 26.*

All four lines are about one kept position, where Alice and Bob used the same basis.

Eve guesses the right basis. Her measurement reads Alice's bit with certainty and leaves the qubit as it was. She resends the same state, and Bob reads the right bit.

$$P = \frac{1}{2}, \quad \text{error } 0$$

Eve guesses the wrong basis. Her result is a coin flip, and the qubit she resends is in one of the two states of her basis. From "The four BB84 states": a state of one basis, measured in the other basis, gives a coin flip. So Bob's bit is wrong half of the time.

$$P = \frac{1}{2}, \quad \text{error } \frac{1}{2}$$

Error rate on the sifted key. Weigh each of the two cases by its probability and add.

$$\text{QBER} = \frac{1}{2} \times 0 + \frac{1}{2} \times \frac{1}{2} = \frac{1}{4}$$

How often Eve's own bit is right. In the right basis she reads Alice's bit for certain. In the wrong basis her result is a coin flip, which is right half of the time.

$$\frac{1}{2} \times 1 + \frac{1}{2} \times \frac{1}{2} = \frac{3}{4}$$

So Eve pays with a 25% error rate on the line, and she still gets only three bits in four right. Intercept and resend is the simplest attack. Cleverer attacks exist, and the security proofs of BB84 cover them. A security proof is a mathematical argument that the final key stays secret whatever the eavesdropper does.

The cell below checks both numbers by trial. It follows 100,000 qubits through an exchange with Eve on every one of them. It builds no circuits. It uses only the two rules from "The four BB84 states": in the same basis the reader gets the bit that was sent, and in the other basis the reader gets a coin flip. `np.where(test, a, b)` takes an entry from `a` where the test is true and from `b` where it is false.

In [ ]:
import numpy as np

trial_rng = np.random.default_rng(2026)     # the seed fixes the random numbers, so the text below stays true
trials = 100_000                            # the number of qubits that Alice sends
alice_bit, alice_basis, eve_basis, bob_basis = trial_rng.integers(0, 2, (4, trials))   # four rows of random 0s and 1s

# Eve measures. Right basis: she reads Alice's bit. Wrong basis: a coin flip. She resends what she read.
eve_bit = np.where(eve_basis == alice_basis, alice_bit, trial_rng.integers(0, 2, trials))
# Bob measures the qubit that Eve resent. Same basis as Eve: he reads her bit. Other basis: a coin flip.
bob_bit = np.where(bob_basis == eve_basis, eve_bit, trial_rng.integers(0, 2, trials))

kept = alice_basis == bob_basis             # sifting: True where Alice and Bob used the same basis
print("positions kept:", kept.sum(), "of", trials)
print("error rate on the sifted key:", round(np.mean(alice_bit[kept] != bob_bit[kept]), 4))
print("how often Eve's bit is right:", round(np.mean(eve_bit[kept] == alice_bit[kept]), 4))

Of the 100,000 positions, 49,885 were kept, about half. On the sifted key the error rate came out as 0.2475, and Eve's bit was right in 0.7525 of the positions. That is one quarter and three quarters, up to the small scatter that any finite number of trials has. Lab 2 repeats this experiment with a circuit for every qubit.

### Compare $n$ bits and she is almost always caught

*Slide 27.*

$$P \left( \text{caught} \right) = 1 - {\left( \frac{3}{4} \right)}^{n}$$

Twenty test bits catch her 99.7% of the time.

How do Alice and Bob find out their error rate? They give up part of the sifted key. They pick some positions at random, tell each other their bits at those positions over the ordinary line, and compare. These are the test bits, and $n$ is how many of them there are. (On this slide $n$ counts test bits. In the lab code the name `n` is the number of qubits that Alice sends.)

On ideal devices with nobody listening, every test bit agrees. If Eve intercepted every qubit, each test bit agrees with probability $\frac{3}{4}$, and a single disagreement gives her away. All $n$ test bits agree, so that she goes unnoticed, with probability $\left( \frac{3}{4} \right)^n$. For $n = 20$ that is $0.75^{20} \approx 0.0032$. She is caught with probability 0.9968, which the slide rounds to 99.7%.

The test bits were revealed in public, so they are thrown away afterward and never used in the key.

#### Derivation: How many test bits are enough?

*Slide 28.*

One test bit agrees (Eve present). The error rate is $\frac{1}{4}$, so a compared bit agrees with probability $1 - \frac{1}{4}$.

$$\frac{3}{4}$$

She survives $n$ tests. Each test bit comes from a different qubit, so the chances multiply.

$$\left( \frac{3}{4} \right)^n$$

To catch her 99% of the time, the chance that she survives must be at most 0.01, that is $0.75^n \leq 0.01$. Take the logarithm of both sides to get $n \ln 0.75 \leq \ln 0.01$, then divide by $\ln 0.75$. That number is negative, so dividing by it turns the inequality around.

$$1 - \left( \frac{3}{4} \right)^n \geq 0.99 \; \Rightarrow \; n \geq \frac{\ln 0.01}{\ln 0.75} \approx 16.01$$

So 17 bits do it: $n$ is a whole number, and 16 falls a little short of 16.01. Twenty bits give

$$1 - 0.75^{20} \approx 0.9968$$

The cell below prints the chance of catching Eve for 16, 17 and `test_bits` test bits and draws it for 1 to 40 test bits. `test_bits` is set to 20 in the first line after the imports.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

test_bits = 20                                   # the number of test bits. Change it, then run this cell and the next code cell again.

for size in (16, 17, test_bits):
    print(size, "test bits: Eve is caught with probability", round(1 - 0.75 ** size, 5))

sizes = np.arange(1, 41)                         # 1 to 40 test bits
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sizes, 1 - 0.75 ** sizes, "o-", color="tab:blue", markersize=3, label="1 - 0.75^n")
ax.axhline(0.99, color="tab:red", ls="--", label="99 percent")
ax.plot(test_bits, 1 - 0.75 ** test_bits, "o", color="black", label="your number of test bits")
ax.set_ylim(0, 1.05)
ax.set_xlabel("number of test bits n")
ax.set_ylabel("chance that Eve is caught")
ax.set_title("Compare n test bits")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.show()

The curve climbs quickly and then flattens. It crosses the 99 percent line between 16 and 17 test bits: 16 bits give 0.98998, a little under 0.99, and 17 give 0.99248. Twenty bits give 0.99683.

The next cell tries the formula out. It simulates 20,000 key exchanges with Eve on every qubit. In each exchange it compares `test_bits` test bits, each of which disagrees with probability one quarter, and it counts the exchanges in which at least one test bit disagrees.

In [ ]:
test_rng = np.random.default_rng(15)             # the seed fixes the random numbers, so the text below stays true
exchanges = 20_000                               # simulated key exchanges, all of them with Eve on every qubit

# One row for each exchange, one column for each test bit. True marks a test bit that disagrees.
disagree = test_rng.random((exchanges, test_bits)) < 0.25
caught = disagree.any(axis=1)                    # True for an exchange with at least one disagreement
print("Eve was caught in", caught.sum(), "of", exchanges, "exchanges, a fraction of", round(caught.mean(), 4))

With 20 test bits Eve was caught in 19,930 of the 20,000 exchanges. That is a fraction of 0.9965, close to the 0.9968 of the formula. Now set `test_bits = 5` in the cell with the curve and run both cells again. The formula then gives $1 - 0.75^5 \approx 0.763$: with so few test bits Eve gets away almost one time in four.

### Noise looks like a small eavesdropper

*Slide 29.*

- A clean channel on a real device still shows a few percent error.
- One intercepted on every qubit shows about 25%.
- The gap between them is what you measure.

Up to here the line and the devices were perfect, and a single wrong test bit meant an eavesdropper. Real devices are noisy: gates and measurements go slightly wrong now and then. A channel is the line that carries the qubits from Alice to Bob, and "clean" means that nobody listens to it. Even a clean channel on a real device shows an error rate of a few percent. Alice and Bob cannot tell whether one particular wrong bit comes from noise or from Eve. What they can do is estimate the error rate from the test bits and ask how large it is.

The standard security proof for BB84 with one-way post-processing (Shor and Preskill, Phys. Rev. Lett. 85, 441, 2000) tolerates error rates up to about 11%. Post-processing is what Alice and Bob do with ordinary messages after the exchange: they correct the remaining errors and shorten the key, so that nothing an eavesdropper may have learned is left in it. One-way means that these messages go in one direction only. Above about 11% the key is thrown away.

This slide is the idea behind Lab 3, which classifies each exchange by its error rate. Lab 3 is also where tonight's challenge starts. The picture on the slide is a number line from 0 to 25 percent, and the cell below redraws it.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.plot([0, 25], [0, 0], color="0.4", lw=1)                               # the number line
ax.plot([1, 5], [0, 0], color="tab:green", lw=9, solid_capstyle="butt")   # a clean channel on a real device
ax.plot([11, 11], [-0.25, 0.25], color="black", ls="--")                  # the discard threshold
ax.plot(25, 0, "o", color="tab:red", markersize=10)                       # intercept and resend
ax.text(3, 0.6, "clean device:\na few percent", ha="center", va="center", fontsize=9)
ax.text(11, 0.6, "discard threshold:\nnear 11 percent", ha="center", va="center", fontsize=9)
ax.text(25, 0.6, "intercept and resend:\n25 percent", ha="center", va="center", fontsize=9)
ax.set_xlim(-2, 31)
ax.set_ylim(-0.4, 1.0)
ax.set_yticks([])
ax.set_xticks([0, 5, 10, 15, 20, 25])
ax.set_xlabel("error rate on the sifted key (percent)")
ax.set_title("Noise or eavesdropper?")
for side in ("left", "right", "top"):
    ax.spines[side].set_visible(False)
plt.show()

The green bar marks a clean channel on a real device, at a few percent. The dashed line is the threshold near 11 percent, and the red dot is intercept and resend at 25 percent. An exchange whose error rate falls left of the dashed line counts as clean. One that falls right of it is treated as intercepted, and its key is thrown away. Lab 3 applies exactly this rule, with the threshold 0.11, to exchanges whose noise lies between 1 and 5 percent, which is where the green bar is drawn.

### Two answers to the quantum threat

*Slide 30.*

- **Quantum key distribution.** Security from physics. Needs a quantum channel and new hardware. Used today on dedicated links.
- **Post-quantum cryptography.** Security from different math problems. Runs on the computers we have. Being deployed at scale now.

The session started with a threat: a large enough quantum computer would break RSA and elliptic curves. There are two answers, and they are different in kind.

Quantum key distribution, QKD, is what you have studied tonight, and BB84 is one of its protocols. Its security rests on physics: measuring disturbs, and an unknown qubit cannot be copied. The price is hardware. The qubits themselves have to travel from Alice to Bob over a quantum channel, such as an optical fiber. The amplifiers of today's networks cannot boost a qubit, because that would mean copying it, which tonight's no-cloning theorem forbids. So a link reaches only as far as enough photons survive, and longer networks pass the key through relay stations that must be trusted. That is why QKD is used today on dedicated links, lines that are built for this one purpose.

Post-quantum cryptography keeps the idea of public-key encryption and replaces the math problem underneath. In place of factoring and discrete logarithms it rests on other problems, chosen because no fast way to solve them is known, neither on a classical computer nor on a quantum one. It needs no qubits at all. It runs on the computers and networks we already have, and it is being deployed at scale now.

NIST, the National Institute of Standards and Technology in the United States, published its first three post-quantum standards in August 2024: FIPS 203 (ML-KEM, for key exchange), FIPS 204 (ML-DSA, for signatures) and FIPS 205 (SLH-DSA, for signatures). FIPS is the name of NIST's series of standards. A signature here is a digital signature, a way to prove who sent a message. Two more were being standardized when this notebook was written: Falcon for signatures and HQC for key exchange (NIST, Post-Quantum Cryptography project page, csrc.nist.gov/projects/post-quantum-cryptography, read on Nov 16, 2026). Look there for the current list before you quote it.

## Part 2: Hands-on

The code in the "Lab" cells is the code on the slides. "File on the slide" gives the name shown above that code on the slide; there is no separate file to download. Run each cell, read what it prints, then change something and run it again. After each lab comes a "Your turn" exercise with a check cell. All of Part 2 runs on a simulator.

### Lab 1: BB84 with nobody listening

*Slide 35.*

File on the slide: `bb84.py`

1. Generate Alice's bits and bases
2. Encode: X for bit 1, H for the X basis
3. Bob applies H before measuring in X
4. Sift, then confirm the error rate is zero

This is the table from "Eight qubits, by hand" with 64 qubits, and with a circuit for every qubit in place of the two rules. In the code a basis is a number: 0 means the Z basis and 1 means the X basis. Line by line:

- `rng = np.random.default_rng()` makes a random number generator. `rng.integers(0, 2, (3, n))` returns three rows of `n` random 0s and 1s: Alice's bits, Alice's bases and Bob's bases. That is step 1.
- The loop handles one qubit at a time. `zip` walks through the three rows together, so `bit`, `a` and `b` are Alice's bit, Alice's basis and Bob's basis for that qubit.
- `QuantumCircuit(1)` makes a circuit with one qubit, which starts in 0. `if bit: qc.x(0)` applies X when the bit is 1, and `if a: qc.h(0)` applies H when Alice's basis is X. That is step 2. In Python the number 1 counts as true and the number 0 as false.
- `if b: qc.h(0)` applies H when Bob's basis is X, and `qc.measure_all()` measures the qubit. The two lines together are Bob's measurement, step 3. The measured bit is stored under the name `meas`.
- `sampler.run([qc], shots=1)` runs the circuit for one shot, because Alice sends each qubit only once. `.result()[0]` is the result for this circuit. `get_bitstrings()` returns a list that holds the measured bit as text, `'0'` or `'1'`, and `int(got[0])` turns it into a number.
- Step 4. `keep = a_bases == b_bases` compares the two rows of bases entry by entry and gives a row of `True` and `False`. Such a row is called a mask. Used as an index, as in `a_bits[keep]`, a mask picks the entries where it is `True`. `wrong` marks the kept positions where Bob's bit differs from Alice's. `keep.sum()` counts the kept positions, and `wrong.mean()` is the fraction of them that are wrong: the error rate.

The slide's code uses the name `sampler` without defining it. The small cell before the lab code provides it: the `StatevectorSampler`, the sampler from Monday's lab. It behaves like an ideal quantum computer, one without noise.

In [ ]:
# Glue for the next cell: the slide's code uses the name sampler without defining it.
# It is the StatevectorSampler from Monday's lab, an ideal simulator. It is left unseeded on purpose:
# a seeded sampler would replay the same random draw for every one-shot circuit.
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

n = 64
rng = np.random.default_rng()
a_bits, a_bases, b_bases = rng.integers(
    0, 2, (3, n))
bob = []
for bit, a, b in zip(a_bits, a_bases, b_bases):
    qc = QuantumCircuit(1)
    if bit: qc.x(0)   # Alice encodes the bit
    if a:   qc.h(0)   # ... in the X basis
    if b:   qc.h(0)   # Bob measures in X
    qc.measure_all()
    r = sampler.run([qc], shots=1).result()[0]
    got = r.data.meas.get_bitstrings()
    bob.append(int(got[0]))
keep = a_bases == b_bases
wrong = a_bits[keep] != np.array(bob)[keep]
print(keep.sum(), "kept, error", wrong.mean())

The cell prints one line, for example `30 kept, error 0.0`. The first number is the length of the sifted key. It changes from run to run, because the bits and the bases are random. About half of the 64 positions are kept, and most runs give between 24 and 40. The error is exactly 0.0 on every run: with nobody listening, and on an ideal simulator, Bob's bit equals Alice's wherever the bases match.

Run the cell a few times and watch the first number move while the second one stays at zero. If you want the same line on every run, give the generator a seed, for example `np.random.default_rng(7)`. A seed is a starting number that fixes the sequence of random numbers.

The next cell prints the first eight qubits of your run in the layout of the table from the seminar.

In [ ]:
letters = ["Z", "X"]                     # the code writes 0 for the Z basis and 1 for the X basis
print("Qubit:        ", *range(1, 9))    # the star prints the entries side by side
print("Alice's bit:  ", *a_bits[:8])
print("Alice's basis:", *[letters[basis] for basis in a_bases[:8]])
print("Bob's basis:  ", *[letters[basis] for basis in b_bases[:8]])
print("Bob's result: ", *bob[:8])
print("Keep?         ", *["y" if flag else "n" for flag in keep[:8]])

Wherever the last row says `y`, the two bases match and Bob's result equals Alice's bit. Where it says `n`, the bases differ and his result is a coin flip, so it may or may not equal her bit. Those positions are thrown away.

#### Your turn 1

Complete the function `sift`. It takes three arrays of the same length: a row of bits and the two rows of bases. An array is a row of numbers from NumPy, the Python library for calculating with many numbers at once, which the code imports as `np`. The lab's `a_bits`, `a_bases` and `b_bases` are arrays. The function returns the bits at the positions where the two bases match. This is the sifting step of the lab, written so that it works for Alice's bits and for Bob's bits alike. Build the mask inside the function, from the function's own arguments `a_bases` and `b_bases`.

In [ ]:
def sift(bits, a_bases, b_bases):
    """Return the bits at the positions where the two bases match."""
    kept = None
    # YOUR CODE HERE
    return kept

In [ ]:
six_bits = np.array([0, 1, 1, 0, 1, 1])             # six test positions
six_a = np.array([1, 1, 0, 0, 1, 0])
six_b = np.array([1, 0, 1, 0, 1, 1])
try:
    got = sift(six_bits, six_a, six_b)
    stopped = ""
except IndexError:
    got = []
    stopped = "An index does not fit the six test positions. Inside the function use only its own arguments bits, a_bases and b_bases, not names from the lab such as keep or a_bits."
except Exception as error:                           # any other error inside sift() becomes a hint
    got = []
    stopped = "sift() stopped on the six test positions with " + type(error).__name__ + ": " + str(error)
assert got is not None, "sift() still returns None. Replace the line YOUR CODE HERE with code that sets kept."
assert not stopped, stopped
is_row = isinstance(got, (list, tuple)) or (isinstance(got, np.ndarray) and got.ndim == 1)
assert is_row, "sift() should return a row of bits, as an array or a list. A single number, such as the count of the kept positions, is not the sifted key."
assert len(got) == 3, "Three of the six test positions have matching bases, so three bits should come back. Build the result from the function's own arguments bits, a_bases and b_bases, not from the lab's a_bits or keep."
assert list(got) == [0, 0, 1], "These are not the bits at the matching positions. Keep a bit only where a_bases equals b_bases, and keep the order."
assert len(sift(a_bits, a_bases, b_bases)) == keep.sum(), "sift() gives the right bits for the six test positions but not for the 64 positions of Lab 1. Compute the result from the arguments bits, a_bases and b_bases, not from fixed numbers."
print("Check passed.")

eight_bits = np.array([1, 0, 0, 1, 1, 0, 1, 0])      # the table from "Eight qubits, by hand"
eight_a = np.array([0, 1, 1, 0, 1, 0, 0, 1])         # Alice's bases Z X X Z X Z Z X, with 0 for Z and 1 for X
eight_b = np.array([0, 0, 1, 1, 1, 0, 1, 0])         # Bob's bases   Z Z X X X Z X Z
print("Key from the eight-qubit table:", "".join(str(bit) for bit in sift(eight_bits, eight_a, eight_b)))
print("Alice's sifted key from Lab 1: ", "".join(str(bit) for bit in sift(a_bits, a_bases, b_bases)))
print("Bob's sifted key from Lab 1:   ", "".join(str(bit) for bit in sift(np.array(bob), a_bases, b_bases)))

### Lab 2: add an eavesdropper

*Slide 36.*

File on the slide: `bb84_eve.py`

1. Insert Eve between Alice and Bob
2. She measures in a random basis, then resends
3. Rerun and watch the error rate move toward 25%

The slide's code is a function, `exchange`. It builds the circuit for one qubit of the exchange, with Eve in the middle if you ask for her.

- `exchange(bit, a, b)` is the circuit of Lab 1: Alice's bit and basis, then Bob's basis and his measurement.
- `exchange(bit, a, b, e)` adds Eve. `e = 0` means that she listens in the Z basis, and `e = 1` that she listens in the X basis. `e=None`, the default, means that nobody listens.
- The circuit has two classical registers of one bit each, named `eve` and `bob`. Eve's measurement is stored in `eve` and Bob's in `bob`. Named registers replace the `measure_all()` of Lab 1, because the two measurements have to be kept apart.
- Eve's three lines are steps 1 and 2 of the slide. To listen in the X basis she applies H and measures. The measurement leaves the qubit in $\lvert 0 \rangle$ or $\lvert 1 \rangle$, whichever she read. The second H turns that back into $\lvert + \rangle$ or $\lvert - \rangle$, so what travels on to Bob is the X-basis state that matches her result. This is the intercept-and-resend attack. To listen in the Z basis she measures and adds no gate.

The cell only defines the function. The cells after it draw one of these circuits and then run many of them.

In [ ]:
from qiskit import (QuantumCircuit,
    QuantumRegister, ClassicalRegister)

def exchange(bit, a, b, e=None):
    q = QuantumRegister(1)
    eve = ClassicalRegister(1, "eve")
    bob = ClassicalRegister(1, "bob")
    qc = QuantumCircuit(q, eve, bob)
    if bit: qc.x(0)
    if a:   qc.h(0)
    if e is not None:     # Eve listens
        if e: qc.h(0)
        qc.measure(0, eve)
        if e: qc.h(0)     # ... and resends
    if b:   qc.h(0)
    qc.measure(0, bob)
    return qc
# mid-circuit measurement: run with Aer's
# sampler, read Bob's bit in result[i].data.bob

The two comment lines at the end of the slide's code say how to run these circuits, and the cells below follow them. A job returns one result for each of its circuits, which is why Lab 1 wrote `.result()[0]`, so Bob's bit for circuit number `i` is in `result[i].data.bob`.

The next cell draws one circuit: `exchange(1, 0, 0, e=1)`.

In [ ]:
exchange(1, 0, 0, e=1).draw("mpl")

Read it from left to right. The label at the left end of the wire, `q0` on a first run, is a name that Qiskit makes up, because the slide's code creates the qubit without a name. The number in it goes up each time `exchange` is called, so it reads differently when you run this cell again. The X gate is Alice sending the bit 1 in the Z basis. The H, the first dial and the second H are Eve: she listens in the X basis, her result goes into the register `eve`, and she resends what she read. The last dial is Bob measuring in the Z basis, into the register `bob`. Alice and Bob used the same basis, so this position is kept. Eve guessed the wrong basis, so Bob reads 0, the wrong bit, half of the time.

Lab 1's `StatevectorSampler` cannot run a circuit with a measurement in the middle. It stops with `QiskitError: 'StatevectorSampler cannot handle mid-circuit measurements'`, which is the first row of the table under "If something breaks". The next cell therefore uses the sampler from Aer, Qiskit's simulator package, which was already used under "Measuring a qubit changes it". It is imported under the name `AerSampler`, as before.

The function `error_rate` in the next cell is not on the slide. It does for a whole exchange what Lab 1 did. It builds one circuit for each qubit with `exchange`, runs all of them in one job with one shot each, reads Bob's bits, sifts, and returns the error rate on the sifted key. Its last argument is Eve's list: for each qubit a basis, or `None` where she does not listen.

In [ ]:
import numpy as np
from qiskit_aer.primitives import SamplerV2 as AerSampler

def error_rate(a_bits, a_bases, b_bases, e_bases):
    """Run one BB84 exchange per qubit and return the error rate on the sifted key."""
    circuits = [exchange(bit, a, b, e)
                for bit, a, b, e in zip(a_bits, a_bases, b_bases, e_bases)]
    result = AerSampler().run(circuits, shots=1).result()      # one job, one shot per circuit
    bob = np.array([int(r.data.bob.get_bitstrings()[0]) for r in result])
    keep = a_bases == b_bases
    return np.mean(a_bits[keep] != bob[keep])

Now run an exchange of 2,000 qubits twice: once with nobody listening, and once with Eve listening to every qubit in a random basis. The bits and the bases come from a random number generator with a seed, so they are the same every time you run the cell. The names in this cell are new (`alice_bits`, not `a_bits`), so that the results of Lab 1 stay as they are.

In [ ]:
seeded_rng = np.random.default_rng(2026)     # a seeded generator: the same bits and bases on every run
many = 2000                                  # the number of qubits in this exchange
alice_bits, alice_bases, bob_bases, eve_bases = seeded_rng.integers(0, 2, (4, many))

print("positions kept:", (alice_bases == bob_bases).sum(), "of", many)
print("error rate with nobody listening:  ", error_rate(alice_bits, alice_bases, bob_bases, [None] * many))
print("error rate with Eve on every qubit:", round(error_rate(alice_bits, alice_bases, bob_bases, eve_bases), 3))

1,024 of the 2,000 positions are kept. With nobody listening the error rate is exactly 0.0, as in Lab 1. With Eve on every qubit it is close to 0.25: one sifted bit in four is wrong, as the seminar predicted.

The bits and bases are fixed by the seed, but the measurements are not, so the last number changes a little when you run the cell again. Almost every run gives a value between 0.22 and 0.28.

Step 3 of the slide: rerun and watch the error rate move toward 25%. How close a run comes to 0.25 depends on the number of bits in the sifted key. The cell below runs four exchanges of growing size, each with Eve on every qubit.

In [ ]:
for size in (64, 256, 1024, 4096):
    bits, bases_a, bases_b, bases_e = seeded_rng.integers(0, 2, (4, size))    # a fresh exchange with this many qubits
    print(size, "qubits: error rate", round(error_rate(bits, bases_a, bases_b, bases_e), 3))

Run this cell several times. The first line, from 64 qubits, jumps around. Only about 32 bits are kept, and with so few the rate can come out anywhere between about 0.10 and 0.40. The last line, from 4,096 qubits, stays within a few hundredths of 0.25.

This is the shot noise from Monday's session. An error rate measured on a limited number of bits scatters around the true value, and the scatter shrinks like one over the square root of the number of bits. Four times as many qubits halve it: it is about 0.08 for 64 qubits and about 0.01 for 4,096.

#### Your turn 2

Eve becomes careful and listens to only about half of the qubits. First predict the error rate on the sifted key. Then build her list `e_half` for the 2,000 qubits of the lab. For each qubit the list holds either a basis chosen at random (0 for Z, 1 for X) or `None` where she lets the qubit pass, and every qubit has an even chance of being listened to. Two tools: `seeded_rng.random()` returns a random number between 0 and 1, and `int(seeded_rng.integers(0, 2))` returns a random 0 or 1. One way to start: go through the positions with a loop over `i`. For each position draw `seeded_rng.random()`, and where it is below 0.5 set `e_half[i]` to a random basis. Every other entry stays `None`.

In [ ]:
e_half = [None] * many          # Eve's list. None means that she does not listen to that qubit.
# YOUR CODE HERE
rate_half = error_rate(alice_bits, alice_bases, bob_bases, e_half)
print("error rate with Eve on about half of the qubits:", round(rate_half, 3))

In [ ]:
listened = [basis for basis in e_half if basis is not None]
assert len(listened) > 0, "Eve is not listening yet. Replace the line YOUR CODE HERE with code that fills e_half."
assert len(e_half) == many, "e_half needs one entry for each of the 2,000 qubits."
assert 0.4 * many < len(listened) < 0.6 * many, "About half of the entries should be a basis, and the rest None."
switches = sum((e_half[i] is None) != (e_half[i + 1] is None) for i in range(many - 1))   # how often listening turns on or off
assert 0.4 * many < switches < 0.6 * many, "Eve should decide at random, for every qubit on its own, whether she listens, with an even chance. Draw a new random number for each qubit."
single = all(np.ndim(basis) == 0 for basis in listened)          # each entry is one value, not a list or an array
assert single and set(listened) == {0, 1}, "Where she listens, Eve picks the Z basis (0) or the X basis (1) at random. Each entry of e_half is the single number 0 or 1, or None."
assert 0.08 < rate_half < 0.17, "The rate should lie between the clean value and the fully intercepted one. Run the cell above again so that rate_half is computed from your list."
print("Check passed. Eve listened to", len(listened), "of the", many, "qubits.")
print("Measured error rate:", round(rate_half, 3), "| compare it with your prediction.")

### Lab 3: a batch of exchanges, some intercepted

*Slide 37.*

File on the slide: `classify.py`

1. Load the batch
2. Compute the error rate of each exchange
3. Choose a threshold
4. Label each one clean or intercepted

This lab is the number line from "Noise looks like a small eavesdropper" with data on it. You are handed the records of twelve key exchanges, and for each one you decide whether somebody listened.

Step 1, the batch. The slide's code expects a variable `batch` and does not define it. The small cell before the lab code makes it. `batch` is a dictionary, a Python collection in which every entry is stored under a name and looked up by that name. It has twelve entries, named `ex01` to `ex12`. Each entry is the record of one exchange of 400 qubits. It is itself a dictionary, with four arrays of 0s and 1s: `alice` (Alice's bits), `a_bases` (Alice's bases), `b_bases` (Bob's bases) and `bob` (Bob's results). As before, 0 stands for the Z basis and 1 for the X basis. Some of the twelve exchanges were intercepted on every qubit. All twelve have a few percent of device noise, which flips some of Bob's bits.

Be clear about where this batch comes from. It is not a recording from a real device. The function `make_batch` in the small cell generates it, with the same two rules as the trial under "Where the 25% comes from", plus the noise. That cell therefore also holds the answer: the number `BATCH_KEY` encodes which exchanges were intercepted. Do not decode it. Work the answer out from the records, as you will have to in the challenge, which uses a different batch.

In [ ]:
# Glue for the next cell: the batch of recorded exchanges that the slide's code calls batch.
# It is generated here, not measured. Twelve exchanges of 400 qubits each, all with device noise.
import numpy as np

BATCH_KEY = 1226          # bit k set: exchange k + 1 was intercepted
def make_batch(seed=2026, n_ex=12, n=400):
    rng = np.random.default_rng(seed)
    batch = {}
    for k in range(n_ex):
        alice, a_bases, b_bases = rng.integers(0, 2, (3, n))
        bits, bases = alice, a_bases
        if (BATCH_KEY >> k) & 1:
            bases = rng.integers(0, 2, n)                                    # Eve's bases
            bits = np.where(bases == a_bases, alice, rng.integers(0, 2, n))  # what she resends
        bob = np.where(b_bases == bases, bits, rng.integers(0, 2, n))
        noise = rng.uniform(0.01, 0.05)                                      # device noise
        bob = bob ^ (rng.random(n) < noise)
        batch[f"ex{k + 1:02d}"] = dict(alice=alice, bob=bob, a_bases=a_bases, b_bases=b_bases)
    return batch

batch = make_batch()
print("exchanges in the batch:", list(batch))
print("arrays in one exchange:", list(batch["ex01"]), "with", len(batch["ex01"]["alice"]), "entries each")

Steps 2 to 4 are the slide's code.

- `qber(ex)` is the sifting and counting of Lab 1, for one exchange `ex`. It keeps the positions where the bases match, compares Alice's bits with Bob's results there, and returns the fraction that differ.
- `threshold = 0.11` is the value from the seminar: the error rate above which a key is thrown away.
- The loop goes through the twelve exchanges. For each one it prints the name, the error rate rounded to three decimals, and the label.

In [ ]:
import numpy as np

def qber(ex):
    keep = ex["a_bases"] == ex["b_bases"]
    a, b = ex["alice"][keep], ex["bob"][keep]
    return np.mean(a != b)

threshold = 0.11
for name, ex in batch.items():
    rate = qber(ex)
    if rate > threshold:
        label = "intercepted"
    else:
        label = "clean"
    print(name, round(rate, 3), label)

The error rates fall into two groups. Seven exchanges lie between 0.005 and 0.051. That is noise alone. Five lie between 0.183 and 0.268: `ex02`, `ex04`, `ex07`, `ex08` and `ex11` were intercepted. Nothing lies in between, so the threshold 0.11 separates the two groups, and so would any other value between 0.051 and 0.183. (The printout drops zeros at the end: `0.03` is 0.030 and `0.25` is 0.250.)

Why are the five high rates not all 0.25? Each exchange has only about 200 sifted bits, and an error rate measured on 200 bits scatters by about 0.03. That is the shot noise of Lab 2 again. Device noise adds a little on top: with a noise rate $p$, an intercepted exchange is expected at $0.25 + p/2$, not at 0.25. Shot noise also puts two clean exchanges, at 0.005 and 0.051, just outside the noise rates of 1 to 5 percent that the batch was made with.

For step 3 of the slide, change `threshold` to 0.04 and run the cell again. Then try 0.2. With 0.04 the noisiest clean exchange is labeled intercepted. With 0.2 one intercepted exchange is labeled clean. Set `threshold` back to 0.11 and run the cell once more before you go on.

The next cell puts the twelve error rates on the number line from the seminar.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.plot([0, 28], [0, 0], color="0.4", lw=1)                                        # the number line
for name, ex in batch.items():
    rate = qber(ex)
    if rate > threshold:
        ax.plot(100 * rate, 0, "s", color="tab:red", markersize=8, alpha=0.7)      # a square: labeled intercepted
    else:
        ax.plot(100 * rate, 0, "o", color="tab:blue", markersize=8, alpha=0.7)     # a circle: labeled clean
ax.plot([100 * threshold, 100 * threshold], [-0.25, 0.25], color="black", ls="--")  # the threshold
ax.plot([], [], "o", color="tab:blue", label="labeled clean")                       # two entries for the legend
ax.plot([], [], "s", color="tab:red", label="labeled intercepted")
ax.legend(loc="upper center", ncol=2, frameon=False)
ax.set_xlim(-2, 31)
ax.set_ylim(-0.4, 1.0)
ax.set_yticks([])
ax.set_xticks([0, 5, 10, 15, 20, 25])
ax.set_xlabel("error rate on the sifted key (percent)")
ax.set_title("The twelve exchanges of the batch")
for side in ("left", "right", "top"):
    ax.spines[side].set_visible(False)
plt.show()

The circles on the left are the exchanges labeled clean, and the squares on the right are the ones labeled intercepted. Markers that lie on top of each other look darker. `ex04` and `ex07` have exactly the same rate, 0.25, so the five intercepted exchanges show as four squares, and the circles overlap as well: the two near 0.5 and 1 percent, and the four between 2 and 3 percent, two of which lie almost on top of each other near 3 percent. The short dashed line is the threshold. The wide empty stretch around it is the gap from the seminar slide: it is what makes the decision safe.

#### Your turn 3

Complete the function `find_suspects`. It takes the batch and a threshold, and it returns a list with the names of the exchanges that you label intercepted: those whose error rate is above the threshold. A name is a piece of text in quotes, such as `"ex02"`. Do not type the names by hand. Let Python collect them, using the lab's `qber` and the function's own argument `threshold`. The last lines of the cell call the function with the lab's `threshold` and store the list in `suspects`.

In [ ]:
def find_suspects(batch, threshold):
    """Return the names of the exchanges whose error rate is above threshold."""
    suspects = None
    # YOUR CODE HERE
    return suspects

suspects = find_suspects(batch, threshold)
print("suspects:", suspects)

In [ ]:
suspects = find_suspects(batch, threshold)   # recompute, so the check always uses the current threshold
assert suspects is not None, "find_suspects() still returns None. Replace the line YOUR CODE HERE with code that builds the list."
is_collection = hasattr(suspects, "__iter__") and hasattr(suspects, "__len__") and not isinstance(suspects, str)
assert is_collection, "suspects should be a list of names. Start from an empty list and append one name at a time."
not_text = sorted({type(entry).__name__ for entry in suspects if not isinstance(entry, str)})
assert not not_text, f"Every entry of suspects should be a name in quotes, such as 'ex01'. Yours holds entries of the type {not_text}. Collect the keys of batch, not the records or their rates."
unknown = [name for name in suspects if name not in batch]
assert not unknown, f"These are not names of exchanges in the batch: {unknown}. Use the keys of batch, such as 'ex01'."
rates = {name: qber(ex) for name, ex in batch.items()}       # the error rate of every exchange
others = [name for name in batch if name not in suspects]    # the exchanges that you call clean
too_low = [name for name in suspects if rates[name] <= threshold]
assert not too_low, "Some of your suspects have a low error rate. Noise alone gives a few percent."
missed = [name for name in others if rates[name] > threshold]
assert not missed, "You missed an intercepted exchange. It shows a rate near 0.25."
assert len(suspects) > 0 and len(others) > 0, "With this threshold all twelve exchanges get the same label. Look at the figure in Lab 3, choose a threshold between the two groups, and run the Lab 3 cell again."
lowest_suspect = min(rates[name] for name in suspects)
highest_other = max(rates[name] for name in others)
assert lowest_suspect - highest_other > 0.1, "Your two groups nearly touch, so the threshold cuts through a group. Look at the figure in Lab 3, move the threshold into the wide empty stretch, and run the Lab 3 cell again."
uses_argument = len(find_suspects(batch, 0.0)) == len(batch) and len(find_suspects(batch, 1.0)) == 0
assert uses_argument, "find_suspects() should use its argument threshold: with threshold 0 every exchange is a suspect, and with threshold 1 none is. Compute the list from the rates, do not type it."
print("Check passed:", len(set(suspects)), "of the", len(batch), "exchanges are labeled intercepted.")
print("lowest error rate among your suspects:", round(lowest_suspect, 3))
print("highest error rate among the others:  ", round(highest_other, 3))

Lab 3 is the starting point of tonight's challenge. A good threshold leaves a wide gap between the two numbers that the check prints. In the challenge the batch is a different one, so read its description in the challenge notebook before you choose a rule.

## What you should see

*Slide 38.*

| Lab | Expected result |
|---|---|
| Lab 1: nobody listening | About half the bits kept, error rate exactly 0 |
| Lab 2: with an eavesdropper | Error rate near 0.25 (25%), steadier with more qubits |
| Lab 3: the batch | Two groups of error rates, one at a few percent and one near 0.25 (25%) |

How near "near 0.25" is depends on the number of qubits. With 64 qubits about 32 bits are kept, and one run can give anything from about 0.10 to 0.40. With 2,000 qubits almost every run gives between 0.22 and 0.28.

## If something breaks

*Slide 39.*

| What you see | What to do |
|---|---|
| StatevectorSampler cannot handle mid-circuit measurements | Use the Aer sampler for the eavesdropper cell |
| Errors appear with nobody listening | Bob applies H only when his basis is X |
| Array shapes do not match | Use the same keep mask on both arrays |
| Different numbers on every run | Expected: the bases are random. Fix the seed to compare runs |
| `AttributeError: 'PrimitiveResult' object has no attribute 'data'` | A job returns one result for each of its circuits. Read Bob's bit from `result[i].data.bob` |
| `NameError: name 'sampler' is not defined`, or the same for `exchange` or `batch` | Run the earlier cells in order, top to bottom. The small cells before Lab 1 and Lab 3 define `sampler` and `batch`, and the Lab 2 code cell defines `exchange` |

The last two rows are not on the slide. They are messages that came up when the notebook was tested. Notes on the four rows from the slide, as they look on the tested versions:

- Row 1 arrives as `QiskitError: 'StatevectorSampler cannot handle mid-circuit measurements'`.
- Row 2: if Bob applies H to every qubit, Lab 1 shows an error rate near 0.25 with nobody listening.
- Row 3 arrives as `ValueError: operands could not be broadcast together with shapes (38,) (64,)` when the keep mask is applied to only one of the two arrays. The two numbers are the sizes of your two arrays. A mask whose length differs from the array it indexes gives `IndexError: boolean index did not match indexed array along axis 0` instead: build the mask from the same arrays that you index.
- Row 4: to fix the seed in Lab 1, write `rng = np.random.default_rng(7)`, with any whole number in the brackets.

## Check yourself

*Slide 31.* On the slides these questions come before the labs.

1. Alice sends $\lvert + \rangle$ and Bob measures in the Z basis. What does he read?
2. Eve intercepts every qubit. What error rate do Alice and Bob see?
3. Which one protects everyday internet traffic at scale: QKD or post-quantum cryptography?

## Tonight in one slide

*Slide 40.*

- **No-cloning.** An unknown qubit cannot be copied.
- **BB84.** Random bases; keep the positions that match.
- **Eavesdropping.** Leaves errors in a quarter of the key; test bits catch her.
- **Two answers.** QKD from physics, post-quantum cryptography from math.

## Challenge 3: Catch the eavesdropper

*Slide 41.*

Spot which BB84 key exchanges were intercepted. The challenge opens Wed, Nov 18 at 6:00 PM and closes Sun, Nov 22 at 11:59 PM, Clemson time (US Eastern).

Start from tonight's notebook: Lab 3 is the starting point. The challenge notebook is posted on the event page when the challenge opens, under Challenges, "Catch the eavesdropper". Submission steps are in the challenge notebook.

## Where to go from here

*Slides 42 and 43.*

- **Tonight.** Classify the batch and start Challenge 3.
- **Try it.** Change the number of test bits and watch the detection rate.
- **Stuck?** Ask in Discord between sessions.

For the second card, go back to "How many test bits are enough?" in Part 1. Change `test_bits` in the cell that draws the curve, then run that cell and the next code cell, which simulates 20,000 exchanges, again.

Physics caught the spy. An eavesdropper who has to measure what she cannot copy leaves errors behind, and tonight you measured them.

- Slides and Challenge 3 are posted tonight.
- Three challenges are open until Sunday.
- Tomorrow at 5:00 PM: Session 4, HPC and quantum machine learning.